# 01 agent framework

**Owner:** Thomas. Original notebook cells: 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

## Thomas: Existing Eye Style Agent Implementation

The cells below contain the existing read-only Crystal/Databricks agent framework. For the capstone, this implementation will be exercised using synthetic or de-identified Crystal-style test data only. Preserve the existing security, permission, and read-only safeguards while adapting the workflow mappings for the Scheduling, Insurance, Optical, and RAG agent demonstrations.

Live Crystal PM API validation and production data connectivity are outside the capstone scope and will be completed separately after the project.

## Agent implementation
The full implementation is embedded so this notebook can be imported by itself. The ZIP also includes editable Python modules and unit tests.

In [ ]:
# Load the extracted reusable framework into the shared notebook namespace.
from pathlib import Path
_framework_path = Path(REPO_ROOT) / "src/agents/framework.py"
exec(compile(_framework_path.read_text(encoding="utf-8"), str(_framework_path), "exec"), globals())


In [ ]:
# Load the authorized vendor contract from a private location. Never commit it.
import os, json
from pathlib import Path
spec_path = Path(os.environ.get("EYE_STYLE_PRIVATE_SPEC", "/Volumes/workspace/default/eye_style_team2/private/crystal_openapi.json"))
if not spec_path.is_file():
    raise FileNotFoundError("Private Crystal API contract missing. See docs/private_setup.md.")
with spec_path.open(encoding="utf-8") as f:
    SPEC = json.load(f)
print("Loaded authorized private API specification.")


## Configure your run

For the capstone, the system will run in demo mode using synthetic or de-identified Crystal-style test data only. No live Crystal PM data or production API calls will be used during the capstone.

The existing authentication, office, scope, and API configuration fields are being preserved for future post-capstone integration work. They are not required for the capstone demonstration.

The capstone will continue to enforce the same role, permission, and read-only safeguards while testing the Scheduling, Insurance, Optical, and RAG workflows with synthetic data.

In [ ]:
# Capstone demo configuration
# The capstone uses synthetic/test Crystal-style data only.

dbutils.widgets.dropdown("data_mode", "demo", ["demo"])
dbutils.widgets.text("model_endpoint", "")
dbutils.widgets.text(
    "question",
    "For September 2026, show frame order counts grouped by status."
)

MODE = dbutils.widgets.get("data_mode")
MODEL = dbutils.widgets.get("model_endpoint").strip()

print("Capstone data mode:", MODE)

In [ ]:
# Capstone synthetic/test-data gateway
# No live Crystal PM connection is used during the capstone.

DEMO_SCOPES = frozenset({
    "health.read",
    "appointments.read",
    "patients.read",
    "insurance.read",
    "orders.read",
    "employees.read",
    "inventory.read",
    "locations.read",
    "files.read",
})

principal = Principal(
    "notebook-demo-owner",
    "demo-office",
    DEMO_SCOPES
)

gateway = CrystalGateway(
    SPEC,
    principal,
    demo=True
)

print("Data mode: demo")
print("Using synthetic Crystal-style test data only.")

## Deterministic demo checks — no LLM required

In [ ]:
# Capstone demo validation
# Uses synthetic Crystal-style data only.

tools = ToolService(gateway)

# Validate synthetic Optical Order workflow
data = tools.read(
    "GetOfficeFrameOrders",
    {
        "startDate": "2026-09-01",
        "endDate": "2026-09-30"
    }
)

report = tools.report(
    data["dataset_id"],
    "September order counts",
    group_by=["status"],
    metrics=[
        {
            "op": "count",
            "alias": "orders"
        }
    ]
)

# Confirm the synthetic fixture contains the expected three orders
assert sum(row["orders"] for row in report["rows"]) == 3

# Confirm write operations remain blocked
try:
    gateway.fetch("CreateInvoice", {})
    raise AssertionError("Write operation was not blocked")
except AgentError:
    pass

print("Capstone demo checks passed:")
print("- Synthetic optical-order data loaded")
print("- Order-status report generated")
print("- Write operations blocked")

display(spark.createDataFrame(report["rows"]))

## Repeatable offline checks
Run these before model inference. They verify tool safeguards and response handling using synthetic records and a fake model client.

In [ ]:
# Offline regression checks
# Capstone version: synthetic/test data only, no live Crystal calls.

from types import SimpleNamespace

checks = []

def check(name, function):
    function()
    checks.append(name)
    print("PASS:", name)

def rejected(function):
    try:
        function()
    except AgentError:
        return
    raise AssertionError("Expected AgentError")


def demo_tools(scopes=DEMO_SCOPES):
    return ToolService(
        CrystalGateway(
            SPEC,
            Principal("test-owner", "test-office", scopes),
            demo=True
        )
    )


def check_orders():
    tools = demo_tools()

    data = tools.read(
        "GetOfficeFrameOrders",
        {
            "startDate": "2026-09-01",
            "endDate": "2026-09-30"
        }
    )

    report = tools.report(
        data["dataset_id"],
        "Orders",
        group_by=["status"],
        metrics=[{"op": "count", "alias": "orders"}]
    )

    assert {
        row["status"]: row["orders"]
        for row in report["rows"]
    } == {
        "InFulfillment": 1,
        "ReadyForDispensing": 1,
        "Dispensed": 1
    }


def check_access():
    tools = demo_tools(frozenset())

    rejected(
        lambda: tools.read(
            "GetOfficeFrameOrders",
            {
                "startDate": "2026-09-01",
                "endDate": "2026-09-30"
            }
        )
    )


def check_overrides_and_writes():
    rejected(
        lambda: demo_tools().read(
            "GetOfficeFrameOrders",
            {"officeId": "another-office"}
        )
    )

    rejected(
        lambda: demo_tools().read(
            "CreateInvoice",
            {}
        )
    )


def check_incomplete_totals():
    tools = ToolService(
        CrystalGateway(
            SPEC,
            Principal("test", "test-office", DEMO_SCOPES),
            demo=True,
            max_rows=1
        )
    )

    data = tools.read("GetOfficeFrameOrders", {})

    assert not data["complete"]

    rejected(
        lambda: tools.report(
            data["dataset_id"],
            "Truncated",
            metrics=[{"op": "count", "alias": "orders"}]
        )
    )


def check_unsupported_demo_source():
    supported = {
        "GetOfficeFrameOrders",
        "GetOfficeContactLensOrders",
        "GetAppointments",
        "GetPatientById",
        "GetHealth"
    }

    tools = demo_tools()

    operation = next(
        name
        for name in tools.gateway.operations
        if name not in supported
    )

    rejected(
        lambda: demo_response(operation, {})
    )


def check_request_local_handles():
    first = demo_tools()

    data = first.read(
        "GetOfficeFrameOrders",
        {
            "startDate": "2026-09-01",
            "endDate": "2026-09-30"
        }
    )

    rejected(
        lambda: demo_tools().report(
            data["dataset_id"],
            "Different request",
            columns=["status"]
        )
    )


def check_csv_protection():
    assert "'=HYPERLINK" in csv_export(
        [{"name": '=HYPERLINK("bad")'}]
    )


for name, function in [
    ("Optical order counts", check_orders),
    ("Denied permissions", check_access),
    ("Office override and writes blocked", check_overrides_and_writes),
    ("Incomplete totals rejected", check_incomplete_totals),
    ("Unsupported demo source rejected", check_unsupported_demo_source),
    ("Request-local dataset handles", check_request_local_handles),
    ("CSV formula protection", check_csv_protection),
]:
    check(name, function)

print(
    f"{len(checks)} offline capstone checks passed. "
    "These validate synthetic workflow safeguards only and do not validate live Crystal data."
)

## Ask the agent

The default model is `system.ai.gpt-oss-120b`, queried through the AI Gateway Chat Completions URL available in the Databricks workspace. Override `model_endpoint` only if the team chooses another compatible model service.

For the capstone, the agent runs in demo mode using synthetic Crystal-style test data only. No live Crystal PM credentials, production data, or production API calls are used.

This cell makes real model inference calls against the synthetic/test workflow data. The model is connected for capstone testing, but the agent is not yet deployed as a production dashboard API.

In [ ]:
# Use the model service successfully tested in this workspace.
# Databricks SDK supplies notebook authentication; no token is printed or embedded.
from databricks.sdk import WorkspaceClient
workspace=WorkspaceClient()
client=workspace.serving_endpoints.get_open_ai_client()
client.base_url=workspace.config.host.rstrip('/')+'/ai-gateway/mlflow/v1'
MODEL=dbutils.widgets.get('model_endpoint').strip() or 'system.ai.gpt-oss-120b'
agent=EyeStyleAgent(client,MODEL,gateway)
last_result=agent.ask(dbutils.widgets.get('question'))
print(last_result['answer'])
for report in last_result['reports']:
    print('Report:',report['title'],'Rows:',len(report['rows']))
    if report['rows']:
        display(spark.createDataFrame([{k:json.dumps(v) if isinstance(v,(dict,list)) else str(v)
            for k,v in row.items()} for row in report['rows'][:100]]))


## Export a generated report

The complete CSV is available in memory at `last_result["reports"][index]["csv"]`; the model sees only a bounded preview. Save only to a private, authorized workspace/Unity Catalog Volume path. The cell below does not write until you provide an export directory.


In [ ]:
EXPORT_DIRECTORY = ""  # Example: /Volumes/<catalog>/<schema>/<volume>/reports
if EXPORT_DIRECTORY and last_result:
    from pathlib import Path
    destination=Path(EXPORT_DIRECTORY)
    if not str(destination).startswith("/Volumes/"):
        raise ValueError("Use an authorized Unity Catalog Volume export path.")
    destination.mkdir(parents=True,exist_ok=True)
    for report in last_result["reports"]:
        (destination/(report["report_id"]+".csv")).write_text(report["csv"],encoding="utf-8")
    print("Saved",len(last_result["reports"]),"reports.")
else:
    print("No export requested.")
